In [1]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class MultiLabelDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['filename']}"
        image = Image.open(img_path).convert("RGB")

        labels = row[1:].values.astype("float32")

        if self.transform:
            image = self.transform(image)

        return image, labels

In [2]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

train_dataset = MultiLabelDataset(
    "dataset/train/_classes.csv",
    "dataset/train",
    transform
)

val_dataset = MultiLabelDataset(
    "dataset/valid/_classes.csv",
    "dataset/valid",
    transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
from pathlib import Path

num_classes = len(train_dataset.df.columns) - 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# ganti head
model.classifier[1] = nn.Linear(1280, num_classes)

model = model.to(device)

In [4]:
criterion = nn.BCEWithLogitsLoss()  # multi-label wajib ini
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [5]:
best_val_loss = float("inf")

for epoch in range(30):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)

            val_outputs = model(val_images)
            val_loss += criterion(val_outputs, val_labels).item()

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "classes": train_dataset.df.columns[1:].tolist(),
    }

    torch.save(checkpoint, checkpoint_dir / "last_model.pth")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(checkpoint, checkpoint_dir / "best_model.pth")
        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f} -> best model disimpan")
    else:
        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

Epoch 1: train_loss=0.2338, val_loss=0.0844 -> best model disimpan
Epoch 2: train_loss=0.0678, val_loss=0.0186 -> best model disimpan
Epoch 3: train_loss=0.0695, val_loss=0.0068 -> best model disimpan
Epoch 4: train_loss=0.0469, val_loss=0.0522
Epoch 5: train_loss=0.0536, val_loss=0.0342
Epoch 6: train_loss=0.0562, val_loss=0.0509
Epoch 7: train_loss=0.0308, val_loss=0.0259
Epoch 8: train_loss=0.0290, val_loss=0.0220
Epoch 9: train_loss=0.0277, val_loss=0.0072
Epoch 10: train_loss=0.0189, val_loss=0.0107
Epoch 11: train_loss=0.0357, val_loss=0.0208
Epoch 12: train_loss=0.0206, val_loss=0.0258
Epoch 13: train_loss=0.0213, val_loss=0.0280
Epoch 14: train_loss=0.0254, val_loss=0.0060 -> best model disimpan
Epoch 15: train_loss=0.0067, val_loss=0.0006 -> best model disimpan
Epoch 16: train_loss=0.0048, val_loss=0.0006 -> best model disimpan
Epoch 17: train_loss=0.0082, val_loss=0.0017
Epoch 18: train_loss=0.0130, val_loss=0.0083
Epoch 19: train_loss=0.0300, val_loss=0.0273
Epoch 20: train_

In [6]:
checkpoint = torch.load(checkpoint_dir / "best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    images = images.to(device)
    outputs = model(images)
    probs = torch.sigmoid(outputs)
    preds = (probs > 0.5).int()

print("Checkpoint terbaik dimuat dari:", checkpoint_dir / "best_model.pth")

Checkpoint terbaik dimuat dari: checkpoints\best_model.pth
